# Escalabilidad, Rendimiento y Costos (Senior)

Objetivos:
- Optimizar rendimiento físico/lógico en DW/Lakehouse.
- Comparar tácticas en Snowflake, BigQuery y Databricks (Delta).
- Gestionar costos (FinOps): presupuestos, tags, chargeback/showback.

<!-- course-nav -->
> Ruta: [Inicio del repositorio](../README.md) | [Índice del nivel](README.md)  
> Anterior: [Gobernanza, Seguridad y Cumplimiento (Senior)](01_gobernanza_seguridad_cumplimiento.ipynb) | Siguiente: [Observabilidad, Lineage y Automatización (Senior)](03_observabilidad_lineage_automatizacion.ipynb)


<!-- course-quality-structure -->
## Prerrequisitos

- Haber revisado los notebooks anteriores de la ruta o contar con experiencia equivalente.
- Conocer los conceptos básicos mencionados en el README del nivel.
- Tener disponible el repositorio local y las dependencias de `requirements.txt` cuando se ejecuten celdas de código.

## Preparación

1. Trabaja desde la raíz del repositorio.
2. Activa tu entorno de Python.
3. Instala dependencias con `pip install -r requirements.txt` si aún no lo hiciste.
4. Ejecuta las celdas en orden para conservar el contexto de variables y resultados.

## Mapa del notebook

1. 1. Modelos de Costo (alto nivel)
2. 1.1. Dimensionamiento Estratégico según 3 V
3. 2. Diseño Físico y Pruning
4. 3. Tácticas por Plataforma
5. 4. Workload Management y Concurrencia
6. 4.1. FinOps Avanzado: Governance y Automation
7. 4.2. Observabilidad Avanzada: Dashboards y SLOs
8. 5. Cost Patterns

## Diagrama de continuidad

```mermaid
flowchart LR
    A["Notebook anterior"] --> B["Escalabilidad, Rendimiento y Costos (Senior)"] --> C["Notebook siguiente"]
```


## 1. Modelos de Costo (alto nivel)

- Snowflake: créditos por compute (virtual warehouses, tamaños), storage aparte; auto-suspend/resume; multi-cluster para concurrencia.
- BigQuery: compute por bytes procesados (on-demand) o slots (reservations); storage separado; particiones/cluster reducen bytes escaneados.
- Databricks: DBUs + costo cloud; clusters autoescalables; Photon (aceleración), Delta Lake (OPTIMIZE/Z-ORDER).

Buenas prácticas de FinOps:
- Etiquetado (tags/labels) por dominio/proyecto.
- Presupuestos y alertas; gobernar tamaños de clúster y horarios.
- Catálogo de costos: dashboards por dominio (chargeback/showback).

## 1.1. Dimensionamiento Estratégico según 3 V

### Volumen: Planificación de Capacidad

**Cálculo de crecimiento:**
```
Capacidad_requerida = Volumen_actual × (1 + tasa_crecimiento)^años × overhead_factor

Donde:
- tasa_crecimiento: típicamente 30-50% anual para datos analíticos
- overhead_factor: 1.3-1.5 para picos y contingencias
```

**Ejemplo:** 100 TB actuales, 40% crecimiento, 3 años, overhead 1.4
```
Año 1: 100 × 1.40 × 1.4 = 196 TB
Año 2: 100 × 1.40² × 1.4 = 274 TB
Año 3: 100 × 1.40³ × 1.4 = 384 TB
```

### Velocidad: Throughput y Latencia

**Dimensionamiento de ingesta:**
- **Batch:** Volumen_total / ventana_tiempo (ej: 1 TB / 1 hora = ~280 MB/s)
- **Streaming:** eventos/seg × bytes/evento × 1.5 (overhead)
- **Micro-batch:** Combinar requisitos de ambos

**SLOs por caso de uso:**
- BI/Reporting: P95 < 5s, P99 < 10s
- Dashboards ejecutivos: P95 < 2s
- APIs de datos: P95 < 500ms
- ML inference: P95 < 100ms

### Variedad: Complejidad de Esquemas

**Impacto en diseño:**
- **Alta variedad** (>10 fuentes heterogéneas): Data Lake/Lakehouse con schema-on-read
- **Baja variedad** (<5 fuentes estructuradas): Data Warehouse con schema-on-write
- **Mixto:** Arquitectura híbrida con capas Bronze (schema-on-read) → Gold (schema-on-write)

In [ ]:
# Calculadora de crecimiento y capacity planning
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def proyectar_capacidad(volumen_inicial_tb, tasa_crecimiento, años, overhead=1.4):
    """Proyecta capacidad necesaria con crecimiento compuesto"""
    proyeccion = []
    for año in range(años + 1):
        capacidad = volumen_inicial_tb * ((1 + tasa_crecimiento) ** año) * overhead
        proyeccion.append({
            'año': 2025 + año,
            'capacidad_tb': round(capacidad, 1),
            'crecimiento_anual_tb': round(capacidad - volumen_inicial_tb * overhead if año == 0 else 
                                          capacidad - proyeccion[-1]['capacidad_tb'], 1)
        })
    return pd.DataFrame(proyeccion)

# Ejemplo: Platform de datos actual 150 TB, crecimiento 45% anual
df_proyeccion = proyectar_capacidad(volumen_inicial_tb=150, tasa_crecimiento=0.45, años=5)

print("=== PROYECCIÓN DE CAPACIDAD (5 AÑOS) ===")
print(df_proyeccion.to_string(index=False))
print(f"\n📈 Capacidad en año 5: {df_proyeccion.iloc[-1]['capacidad_tb']:.1f} TB")
print(f"📊 Crecimiento total: {(df_proyeccion.iloc[-1]['capacidad_tb'] / df_proyeccion.iloc[0]['capacidad_tb'] - 1) * 100:.0f}%")

# Visualización
plt.figure(figsize=(12, 6))
plt.plot(df_proyeccion['año'], df_proyeccion['capacidad_tb'], marker='o', linewidth=2, markersize=8)
plt.fill_between(df_proyeccion['año'], 0, df_proyeccion['capacidad_tb'], alpha=0.3)
plt.xlabel('Año')
plt.ylabel('Capacidad (TB)')
plt.title('Proyección de Capacidad de Storage (5 años)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Diseño Físico y Pruning

Técnicas comunes:
- Particionamiento temporal/geográfico.
- Clustering/ordenamiento (sort keys, Z-ORDER).
- Formatos columnares (Parquet/ORC) y compresión adecuada.
- Estadísticas actualizadas y materialized views seleccionadas.
- Proyección de columnas (evitar SELECT *).

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(1)
n = 200_000
df = pd.DataFrame({
    'fecha': pd.date_range('2025-01-01', periods=n, freq='min'),
    'region': np.random.choice(['Norte', 'Sur', 'Este', 'Oeste'], size=n),
    'sku': np.random.randint(1, 5000, size=n),
    'precio': np.random.uniform(5, 500, size=n),
    'cantidad': np.random.randint(1, 10, size=n)
})
df['importe'] = df['precio'] * df['cantidad']

sample_all = df.head(1000).copy()
sample_proj = df[['region', 'importe']].head(1000).copy()
print('Columnas all:', sample_all.columns.tolist())
print('Columnas proyección:', sample_proj.columns.tolist())

mask = (df['fecha'] >= '2025-02-01') & (df['fecha'] < '2025-03-01')
febrero = df.loc[mask, ['fecha', 'region', 'importe']].head(3)
print('\nEjemplo pruning febrero:')
print(febrero)


## 3. Tácticas por Plataforma

Snowflake:
- Auto-suspend/resume de warehouses; dimensionar por carga, escalar a picos.
- Result/metadata caching; clustering keys en tablas grandes.
- Multi-cluster warehouses para concurrencia; query acceleration opcional.

BigQuery:
- Tablas particionadas (por fecha o ingestion-time) y clustering (hasta 4 cols).
- Materialized views para agregaciones repetidas; BI Engine para dashboards.
- Evitar SELECT *; usar filtros de partición; control de bytes escaneados.

Databricks (Delta):
- Photon; Auto-Optimize; OPTIMIZE + ZORDER(col1,col2) para locality.
- Caching y AQE (Adaptive Query Execution) en Spark.
- Autoscaling y políticas de clúster; spot/low-priority cuando aplique.

## 4. Workload Management y Concurrencia

- Separar workloads: ingestión, transformaciones, BI/serving (compute aislado).
- Programación inteligente (ventanas nocturnas, micro-batch).
- Pooling de recursos y límites de cuotas; SLAs/SLOs por cola.

## 4.1. FinOps Avanzado: Governance y Automation

### Framework FinOps para Datos

**Fases del ciclo FinOps:**
1. **Inform (Visibilidad):** Dashboards de costos en tiempo real
2. **Optimize (Optimización):** Identificar y ejecutar oportunidades de ahorro
3. **Operate (Operación):** Automatizar políticas y alertas

### Estrategias de Optimización de Costos

#### A. Chargeback vs Showback

**Showback (informativo):**
- Reportes mensuales de consumo por equipo/proyecto
- No hay facturación real, solo visibilidad
- Útil para iniciar cultura de cost awareness

**Chargeback (accountability):**
- Facturación interna real a cada equipo
- Requiere aprobación de presupuestos
- Fomenta ownership y optimización proactiva

**Implementación:**
```yaml
modelo_costos:
  unidad_negocio: Marketing
  centro_costos: CC-1234
  asignacion:
    storage: uso_real × tarifa_interna
    compute: horas_cluster × tarifa_hora
    egress: gb_transferidos × tarifa_gb
  presupuesto_mensual: 25000 USD
  alertas:
    - umbral: 50%  # $12,500
    - umbral: 80%  # $20,000
    - umbral: 100% # $25,000 (bloqueo automático)
```

#### B. Cost Anomaly Detection

Detectar spikes inesperados de costo:
- **Desviación estándar:** costo_diario > promedio + 2σ
- **Machine Learning:** Prophet, ARIMA para forecasting
- **Acciones:** Alertas automáticas, rollback de cambios recientes

#### C. Reserved Instances / Committed Use Discounts

**Cuándo comprar:**
- Workloads predecibles con > 70% utilización continua
- Descuentos: 30-70% vs on-demand

**Trade-offs:**
- ✅ Ahorro significativo
- ❌ Falta flexibilidad, compromiso 1-3 años
- ❌ Riesgo de sobre-provisionar

#### D. Spot Instances / Preemptible VMs

**Casos de uso:**
- Procesamiento batch no crítico
- Backfills históricos
- Ambientes de desarrollo/test

**Ahorro:** 60-90% vs on-demand
**Riesgo:** Pueden interrumpirse con 30s-2min aviso

In [ ]:
# Sistema de detección de anomalías de costo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Generar datos históricos de costo (90 días)
np.random.seed(42)
fechas = pd.date_range(end=datetime.now(), periods=90, freq='D')

# Patrón base: costo normal con leve tendencia creciente + variación semanal
costo_base = 2000
tendencia = np.linspace(0, 500, 90)
variacion_semanal = 300 * np.sin(np.arange(90) * 2 * np.pi / 7)
ruido = np.random.normal(0, 150, 90)
costos_historicos = costo_base + tendencia + variacion_semanal + ruido

# Inyectar anomalías
costos_historicos[45] = 6500  # Spike día 45
costos_historicos[70] = 7200  # Spike día 70

df_costos = pd.DataFrame({
    'fecha': fechas,
    'costo_usd': costos_historicos
})

# Calcular estadísticas para detección
media_movil = df_costos['costo_usd'].rolling(window=7).mean()
std_movil = df_costos['costo_usd'].rolling(window=7).std()
umbral_superior = media_movil + 2 * std_movil
umbral_inferior = media_movil - 2 * std_movil

# Detectar anomalías
df_costos['es_anomalia'] = (df_costos['costo_usd'] > umbral_superior) | (df_costos['costo_usd'] < umbral_inferior)
anomalias = df_costos[df_costos['es_anomalia']]

# Visualización
fig, ax = plt.subplots(figsize=(16, 6))

# Costo diario
ax.plot(df_costos['fecha'], df_costos['costo_usd'], label='Costo Diario', 
        color='steelblue', linewidth=1.5)

# Media móvil y bandas de confianza
ax.plot(df_costos['fecha'], media_movil, label='Media Móvil (7 días)', 
        color='green', linestyle='--', linewidth=2)
ax.fill_between(df_costos['fecha'], umbral_inferior, umbral_superior, 
                alpha=0.2, color='green', label='Banda de confianza (±2σ)')

# Marcar anomalías
if len(anomalias) > 0:
    ax.scatter(anomalias['fecha'], anomalias['costo_usd'], 
              color='red', s=200, marker='X', zorder=5, label='Anomalías Detectadas')

ax.set_xlabel('Fecha')
ax.set_ylabel('Costo (USD)')
ax.set_title('Sistema de Detección de Anomalías de Costo', fontsize=14, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Reporte de anomalías
print("=" * 80)
print("ALERTAS DE ANOMALÍAS DE COSTO")
print("=" * 80)
if len(anomalias) > 0:
    for idx, row in anomalias.iterrows():
        desviacion = row['costo_usd'] - media_movil.iloc[idx]
        pct_desviacion = (desviacion / media_movil.iloc[idx]) * 100
        print(f"🚨 [{row['fecha'].date()}] Costo: ${row['costo_usd']:,.2f} "
              f"(+${desviacion:,.2f}, +{pct_desviacion:.1f}% vs promedio)")
else:
    print("✅ No se detectaron anomalías en el período")

print(f"\n📊 Costo promedio diario: ${df_costos['costo_usd'].mean():,.2f}")
print(f"💰 Costo total (90 días): ${df_costos['costo_usd'].sum():,.2f}")

## 4.2. Observabilidad Avanzada: Dashboards y SLOs

### Arquitectura de Observabilidad

**Stack recomendado:**
```
Recolección → Procesamiento → Almacenamiento → Visualización → Alerting
   ↓               ↓                ↓               ↓              ↓
Telegraf      Prometheus         TimescaleDB     Grafana       PagerDuty
FluentBit     Datadog Agent      InfluxDB        Datadog       Opsgenie
OpenTelemetry Vector             Elasticsearch   Kibana        Slack
```

### Métricas Clave por Capa

#### Capa de Ingesta
- **Throughput:** Eventos/segundo, GB/hora
- **Lag:** Tiempo entre producción y consumo (Kafka consumer lag)
- **Error rate:** % mensajes fallidos
- **Backpressure:** Queue depth, tamaño buffer

#### Capa de Procesamiento
- **Job duration:** Tiempo de ejecución batch/streaming
- **CPU/Memory utilization:** % uso por worker
- **Data skew:** Distribución desbalanceada entre particiones
- **Shuffle size:** Bytes intercambiados entre nodos (Spark)

#### Capa de Serving
- **Query latency:** P50, P95, P99
- **Query throughput:** Queries/segundo
- **Cache hit rate:** % queries servidas desde cache
- **Concurrent queries:** Número de queries en ejecución

### SLOs (Service Level Objectives)

**Definición de SLO:**
```
SLO = SLI (indicador) + Target (objetivo) + Measurement Window (ventana)

Ejemplo:
- SLI: P95 de latencia de queries
- Target: < 3 segundos
- Window: 30 días
- SLO: "P95 query latency < 3s en 99.5% de las mediciones en 30 días"
```

**Error Budget:**
```
Error Budget = (1 - SLO) × total_requests

Ejemplo: SLO 99.9% uptime mensual
Error Budget = 0.1% × 30 días = 43 minutos de downtime permitido
```

### Golden Signals para Datos

1. **Latency:** Tiempo de respuesta
2. **Traffic:** Volumen de datos/queries
3. **Errors:** Tasa de fallos
4. **Saturation:** Utilización de recursos (CPU, memoria, disco)

In [ ]:
# Dashboard de Observabilidad Completo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# Generar métricas sintéticas para el último mes
np.random.seed(100)
horas = pd.date_range(end=datetime.now(), periods=24*30, freq='h')

metricas = pd.DataFrame({
    'timestamp': horas,
    # Ingesta
    'throughput_mbps': np.random.gamma(80, 2, len(horas)) + 50 * np.sin(np.arange(len(horas)) * 2 * np.pi / 24),
    'kafka_lag_seg': np.random.exponential(5, len(horas)),
    # Procesamiento
    'cpu_util_pct': np.random.beta(6, 4, len(horas)) * 100,
    'memory_util_pct': np.random.beta(7, 3, len(horas)) * 100,
    # Serving
    'query_latency_p95_ms': np.random.gamma(15, 100, len(horas)),
    'queries_per_min': np.random.poisson(150, len(horas)),
    'error_rate_pct': np.random.exponential(0.3, len(horas)),
})

# Inyectar algunos eventos
metricas.loc[300:310, 'query_latency_p95_ms'] = 4500  # Spike de latencia
metricas.loc[500:505, 'error_rate_pct'] = 5.0         # Spike de errores

# Calcular SLO compliance
SLO_LATENCY_MS = 2000  # P95 < 2s
SLO_ERROR_RATE = 1.0   # < 1%
SLO_CPU = 85           # < 85%

metricas['slo_latency_ok'] = metricas['query_latency_p95_ms'] < SLO_LATENCY_MS
metricas['slo_error_ok'] = metricas['error_rate_pct'] < SLO_ERROR_RATE
metricas['slo_cpu_ok'] = metricas['cpu_util_pct'] < SLO_CPU

# Crear dashboard completo
fig = plt.figure(figsize=(20, 12))
gs = GridSpec(3, 3, figure=fig, hspace=0.3, wspace=0.3)

# Row 1: Ingesta
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(metricas['timestamp'], metricas['throughput_mbps'], linewidth=0.8, color='steelblue')
ax1.set_ylabel('Throughput (Mbps)')
ax1.set_title('Ingesta: Throughput')
ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(metricas['timestamp'], metricas['kafka_lag_seg'], linewidth=0.8, color='coral')
ax2.axhline(y=30, color='red', linestyle='--', label='Critical: 30s')
ax2.set_ylabel('Lag (segundos)')
ax2.set_title('Ingesta: Kafka Consumer Lag')
ax2.legend()
ax2.grid(True, alpha=0.3)

ax3 = fig.add_subplot(gs[0, 2])
throughput_24h = metricas.tail(24)['throughput_mbps']
ax3.bar(range(24), throughput_24h.values, color='teal', alpha=0.7)
ax3.set_xlabel('Hora del día')
ax3.set_ylabel('Throughput (Mbps)')
ax3.set_title('Ingesta: Patrón últimas 24h')
ax3.grid(axis='y', alpha=0.3)

# Row 2: Procesamiento
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(metricas['timestamp'], metricas['cpu_util_pct'], linewidth=0.8, color='orange', label='CPU')
ax4.axhline(y=SLO_CPU, color='red', linestyle='--', label=f'SLO: {SLO_CPU}%')
ax4.set_ylabel('Utilización (%)')
ax4.set_title('Procesamiento: CPU Utilization')
ax4.legend()
ax4.grid(True, alpha=0.3)

ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(metricas['timestamp'], metricas['memory_util_pct'], linewidth=0.8, color='purple')
ax5.axhline(y=90, color='red', linestyle='--', label='Critical: 90%')
ax5.set_ylabel('Utilización (%)')
ax5.set_title('Procesamiento: Memory Utilization')
ax5.legend()
ax5.grid(True, alpha=0.3)

ax6 = fig.add_subplot(gs[1, 2])
slo_compliance = [
    metricas['slo_cpu_ok'].sum() / len(metricas) * 100,
    metricas['slo_latency_ok'].sum() / len(metricas) * 100,
    metricas['slo_error_ok'].sum() / len(metricas) * 100
]
colors_slo = ['green' if x >= 99.5 else 'orange' if x >= 99 else 'red' for x in slo_compliance]
ax6.barh(['CPU < 85%', 'Latency < 2s', 'Error < 1%'], slo_compliance, color=colors_slo)
ax6.axvline(x=99.5, color='green', linestyle='--', label='SLO Target: 99.5%')
ax6.set_xlabel('Compliance (%)')
ax6.set_title('SLO Compliance (30 días)')
ax6.legend()
ax6.grid(axis='x', alpha=0.3)

# Row 3: Serving
ax7 = fig.add_subplot(gs[2, 0])
ax7.plot(metricas['timestamp'], metricas['query_latency_p95_ms'], linewidth=0.8, color='darkgreen')
ax7.axhline(y=SLO_LATENCY_MS, color='red', linestyle='--', label=f'SLO: {SLO_LATENCY_MS}ms')
ax7.set_ylabel('Latencia P95 (ms)')
ax7.set_title('Serving: Query Latency P95')
ax7.legend()
ax7.grid(True, alpha=0.3)

ax8 = fig.add_subplot(gs[2, 1])
ax8.plot(metricas['timestamp'], metricas['queries_per_min'], linewidth=0.8, color='dodgerblue')
ax8.set_ylabel('Queries/minuto')
ax8.set_title('Serving: Query Throughput')
ax8.grid(True, alpha=0.3)

ax9 = fig.add_subplot(gs[2, 2])
ax9.plot(metricas['timestamp'], metricas['error_rate_pct'], linewidth=0.8, color='crimson')
ax9.axhline(y=SLO_ERROR_RATE, color='red', linestyle='--', label=f'SLO: {SLO_ERROR_RATE}%')
ax9.set_ylabel('Error Rate (%)')
ax9.set_title('Serving: Error Rate')
ax9.legend()
ax9.grid(True, alpha=0.3)

fig.suptitle('Dashboard de Observabilidad - Plataforma de Datos', 
             fontsize=18, fontweight='bold', y=0.995)
plt.show()

# Reporte de SLOs
print("=" * 80)
print("REPORTE DE SLOs (ÚLTIMOS 30 DÍAS)")
print("=" * 80)
print(f"\n✅ CPU Utilization < {SLO_CPU}%:")
print(f"   Compliance: {slo_compliance[0]:.2f}% ({'CUMPLE' if slo_compliance[0] >= 99.5 else 'NO CUMPLE'})")
print(f"   Violaciones: {(~metricas['slo_cpu_ok']).sum()} de {len(metricas)} mediciones")

print(f"\n✅ Query Latency P95 < {SLO_LATENCY_MS}ms:")
print(f"   Compliance: {slo_compliance[1]:.2f}% ({'CUMPLE' if slo_compliance[1] >= 99.5 else 'NO CUMPLE'})")
print(f"   Violaciones: {(~metricas['slo_latency_ok']).sum()} de {len(metricas)} mediciones")

print(f"\n✅ Error Rate < {SLO_ERROR_RATE}%:")
print(f"   Compliance: {slo_compliance[2]:.2f}% ({'CUMPLE' if slo_compliance[2] >= 99.5 else 'NO CUMPLE'})")
print(f"   Violaciones: {(~metricas['slo_error_ok']).sum()} de {len(metricas)} mediciones")

# Error budget
error_budget_pct = 100 - 99.5
error_budget_hours = len(metricas) * (error_budget_pct / 100)
error_budget_used = (~metricas['slo_latency_ok']).sum()
print(f"\n📊 Error Budget (Latency SLO):")
print(f"   Presupuesto: {error_budget_hours:.1f} horas ({error_budget_pct}%)")
print(f"   Consumido: {error_budget_used} horas")
print(f"   Restante: {error_budget_hours - error_budget_used:.1f} horas")

## 5. Cost Patterns

- Auto-suspend en Snowflake; reservations/slots adecuados en BigQuery.
- Cerrar clústeres o escalar a cero en Databricks fuera de horario.
- Seleccionar tamaños de archivo óptimos (128–512 MB) para Parquet.
- Catálogo de costos con tags/labels + reportes periódicos.

## 6. Ejercicio

Escenario: dashboard con latencia P95 < 2s (30 días de datos).
- Propón partición/clustering por plataforma.
- Define compute (tamaño warehouse/slots/cluster) y horarios.
- Lista 5 tácticas de reducción de costo sin degradar SLO.

In [ ]:
plantilla = '''
# Solución (borrador)
Partición/Clustering: [Snowflake/BigQuery/Databricks]
Compute: [WH size / slots / cluster size]
Tácticas costo: [lista]
Monitoreo: [métricas de costo y rendimiento]
'''
print(plantilla)

## 7. Resumen

- Diseña para pruning y proyección; evita SELECT *.
- Ajusta compute a patrones reales y automatiza suspensión.
- Mide costo y rendimiento con métricas y alertas.

## 8. Ejercicio Integrador Avanzado

**Escenario:** Migración multi-cloud y optimización de costos

Tu empresa opera una plataforma de datos en AWS con los siguientes componentes:
- **Data Lake (S3):** 500 TB (300 TB hot, 200 TB cold)
- **Compute (EMR):** 10 clusters m5.4xlarge (16 vCPUs, 64GB RAM) × 8h/día promedio
- **Data Warehouse (Redshift):** dc2.8xlarge (32 vCPUs, 244GB RAM) × 24/7
- **Egress:** 20 TB/mes hacia partners externos
- **Crecimiento proyectado:** 50% anual en volumen, 30% en compute

**Costos actuales mensuales:**
- Storage: $6,900
- Compute EMR: $12,000
- Redshift: $48,960
- Egress: $1,800
- **TOTAL: $69,660/mes**

**Mandato ejecutivo:** Reducir costos 30% en 6 meses sin degradar SLOs

**SLOs actuales:**
- P95 query latency < 5s
- Data freshness < 15 min
- Uptime 99.9%

**Tareas:**

### 1. Análisis de Capacity Planning (20%)
- Proyectar storage y compute para 3 años
- Identificar componentes que excederán capacidad actual
- Calcular inversión requerida sin optimización

### 2. Estrategia de Optimización Multi-Dimensional (30%)
- **Storage:** Proponer tiering strategy (hot/warm/cold/archive)
- **Compute:** Estrategias de right-sizing, auto-scaling, spot instances
- **Arquitectura:** Evaluar migración parcial a BigQuery/Snowflake/Synapse

### 3. Plan FinOps Detallado (30%)
- Comparar TCO (Total Cost of Ownership) de 3 opciones arquitectónicas
- Diseñar modelo de chargeback por dominio de datos
- Implementar sistema de alertas y cost anomaly detection
- Definir métricas de eficiencia ($/GB almacenado, $/TB procesado)

### 4. Observabilidad y Governance (20%)
- Diseñar dashboard ejecutivo con métricas clave
- Definir 5 SLOs críticos con error budgets
- Proponer estrategia de alerting (niveles, canales, on-call)
- Establecer procesos de revisión trimestral de costos

**Entregables:**
1. Documento ejecutivo (5-7 páginas) con análisis y recomendaciones
2. Modelo financiero en Excel/Python con proyecciones 3 años
3. Diagrama arquitectónico propuesto con costos desglosados
4. Plan de implementación por fases (6 meses)

<!-- course-quality-close -->
## Ejercicio de cierre

Revisa el caso trabajado en el notebook y documenta una decisión de arquitectura: contexto, opción elegida, alternativas descartadas, riesgos y métricas de validación.

## Evaluación

- El notebook se ejecuta en orden sin depender de estado oculto.
- Las decisiones técnicas están justificadas con trade-offs claros.
- Los resultados, diagramas o tablas apoyan la explicación y no reemplazan el razonamiento.
- Las referencias usadas son oficiales, primarias o están incluidas en la bibliografía del curso.

## Referencias

- FinOps Framework: https://www.finops.org/framework/
- Referencias del curso: ../04-recursos/referencias.md
